# W05 — Structured Content Archetype Clustering

**Pipeline:** one row per content → feature preparation → clustering → validation → archetype interpretation → actions → self-check

This notebook connects to the FlyRank warehouse via DuckDB, builds a single content-level modeling table, and applies K-Means clustering to discover interpretable content archetypes.

## 1. Data + Modeling Grain

Connect to the warehouse, register the four FlyRank tables, and build one row per content item (`model_df`).

In [ ]:

import os
import sys
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display

# Configure display options for clean output
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

print('✅ Core libraries loaded successfully.')

In [ ]:
# Cell 1: Authenticate with Hugging Face via DuckDB Secret
# Load environment variables from .env file
load_dotenv(override=True)
HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError('❌ HF_TOKEN not found! Please add HF_TOKEN=hf_... to your .env file.')

# Initialize in-memory DuckDB connection
con = duckdb.connect()

# Create Hugging Face authentication secret in DuckDB
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

print('✅ DuckDB connected and Hugging Face secret registered successfully.')

In [ ]:
# Cell 2: Define Table Mappings and Register DuckDB Views
# Base URI for Hugging Face dataset repository
rel = 'hf://datasets/FlyRank/internship-warehouse'

# Exact path mapping based on repo structure:
# - dim_clients, dim_content, and fact_content_query_90d are single parquet files
# - fact_content_daily_performance is partitioned by month (month=YYYY-MM/)
table_paths = {
    'dim_clients': f'{rel}/dim_clients.parquet',
    'dim_content': f'{rel}/dim_content.parquet',
    'fact_content_daily_performance': f'{rel}/fact_content_daily_performance/**/*.parquet',
    'fact_content_query_90d': f'{rel}/fact_content_query_90d.parquet'
}

# Create views in DuckDB for in-place querying
for name, path in table_paths.items():
    print(f'Registering view: {name} ...')
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{path}')")

print('\n✅ All 4 tables/views registered successfully in DuckDB!')


In [ ]:
content_info = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT content_hash_id) AS unique_content,
        COUNT(DISTINCT client_hash_id) AS unique_clients
    FROM dim_content
""").fetchdf()

display(content_info)

In [ ]:
query_info = con.execute("""
    SELECT
        MIN(window_start) AS min_window_start,
        MAX(window_end) AS max_window_end,
        COUNT(*) AS row_count,
        COUNT(DISTINCT content_hash_id) AS content_count,
        COUNT(DISTINCT query_hash_id) AS query_count
    FROM fact_content_query_90d
""").fetchdf()

display(query_info)

In [ ]:
latest_date = con.execute("""
    SELECT MAX(report_date)
    FROM fact_content_daily_performance
""").fetchone()[0]

print("Latest performance date:", latest_date)

In [ ]:
window_start = pd.Timestamp(latest_date) - pd.Timedelta(days=89)

print("Modeling window:")
print("Start:", window_start.date())
print("End:", pd.Timestamp(latest_date).date())

In [ ]:
client_history = con.execute("""
    SELECT
        COUNT(*) AS total_clients,
        SUM(
            CASE
                WHEN gsc_data_start <= DATE '2026-04-02'
                THEN 1
                ELSE 0
            END
        ) AS clients_with_full_gsc_window,
        SUM(
            CASE
                WHEN ga4_data_start <= DATE '2026-04-02'
                THEN 1
                ELSE 0
            END
        ) AS clients_with_full_ga4_window
    FROM dim_clients
""").fetchdf()

display(client_history)

In [ ]:
daily_features_final = con.execute(f"""
    SELECT
        d.client_hash_id,
        d.content_hash_id,

        MIN(d.report_date) AS first_report_date,
        MAX(d.report_date) AS last_report_date,
        COUNT(*) AS days_observed,

        SUM(
            CASE
                WHEN d.gsc_data_available
                THEN COALESCE(d.gsc_impressions, 0)
                ELSE 0
            END
        ) AS impressions_90d,

        SUM(
            CASE
                WHEN d.gsc_data_available
                THEN COALESCE(d.gsc_clicks, 0)
                ELSE 0
            END
        ) AS clicks_90d,

        CASE
            WHEN SUM(
                CASE
                    WHEN d.gsc_data_available
                    THEN COALESCE(d.gsc_impressions, 0)
                    ELSE 0
                END
            ) > 0
            THEN
                SUM(
                    CASE
                        WHEN d.gsc_data_available
                        THEN COALESCE(d.gsc_clicks, 0)
                        ELSE 0
                    END
                ) * 1.0
                /
                SUM(
                    CASE
                        WHEN d.gsc_data_available
                        THEN COALESCE(d.gsc_impressions, 0)
                        ELSE 0
                    END
                )
            ELSE NULL
        END AS ctr_90d,

        CASE
            WHEN SUM(
                CASE
                    WHEN d.gsc_data_available
                    THEN COALESCE(d.gsc_impressions, 0)
                    ELSE 0
                END
            ) > 0
            THEN
                SUM(
                    CASE
                        WHEN d.gsc_data_available
                        THEN COALESCE(d.gsc_sum_position, 0)
                        ELSE 0
                    END
                ) * 1.0
                /
                SUM(
                    CASE
                        WHEN d.gsc_data_available
                        THEN COALESCE(d.gsc_impressions, 0)
                        ELSE 0
                    END
                )
            ELSE NULL
        END AS avg_position_90d,

        SUM(
            CASE
                WHEN d.ga4_data_available
                THEN COALESCE(d.ga4_pageviews, 0)
                ELSE 0
            END
        ) AS pageviews_90d,

        SUM(
            CASE
                WHEN d.ga4_data_available
                THEN COALESCE(d.ga4_sessions, 0)
                ELSE 0
            END
        ) AS sessions_90d,

        SUM(
            CASE
                WHEN d.ga4_data_available
                THEN COALESCE(d.ga4_users, 0)
                ELSE 0
            END
        ) AS users_90d,

        SUM(
            CASE
                WHEN d.ga4_data_available
                THEN COALESCE(d.ga4_engaged_sessions, 0)
                ELSE 0
            END
        ) AS engaged_sessions_90d,

        SUM(
            CASE
                WHEN d.ga4_data_available
                THEN COALESCE(d.sessions_ai, 0)
                ELSE 0
            END
        ) AS ai_sessions_90d,

        SUM(
            CASE
                WHEN d.ga4_data_available
                THEN COALESCE(d.scroll_events, 0)
                ELSE 0
            END
        ) AS scroll_events_90d,

        SUM(
            CASE
                WHEN d.gsc_data_available
                THEN COALESCE(d.sessions_organic, 0)
                ELSE 0
            END
        ) AS organic_sessions_90d

    FROM fact_content_daily_performance d

    WHERE d.report_date BETWEEN
        DATE '{window_start.date()}'
        AND DATE '{latest_date}'

    GROUP BY
        d.client_hash_id,
        d.content_hash_id
""").fetchdf()

print("Final daily features:", daily_features_final.shape)
display(daily_features_final.head())

In [ ]:
query_features_final = con.execute("""
    SELECT
        q.client_hash_id,
        q.content_hash_id,

        COUNT(DISTINCT q.query_hash_id) AS query_count_90d,

        SUM(COALESCE(q.impressions_90d, 0)) AS query_impressions_90d,
        SUM(COALESCE(q.clicks_90d, 0)) AS query_clicks_90d,

        SUM(COALESCE(q.impressions_last30, 0)) AS query_impressions_last30,
        SUM(COALESCE(q.clicks_last30, 0)) AS query_clicks_last30,

        SUM(COALESCE(q.impressions_prev30, 0)) AS query_impressions_prev30,
        SUM(COALESCE(q.clicks_prev30, 0)) AS query_clicks_prev30,

        ANY_VALUE(q.content_total_impressions_90d)
            AS content_total_impressions_90d,

        ANY_VALUE(q.content_visible_query_count)
            AS content_visible_query_count,

        ANY_VALUE(q.rare_query_count)
            AS rare_query_count,

        ANY_VALUE(q.rare_impressions_share)
            AS rare_impressions_share,

        ANY_VALUE(q.anonymized_impressions_share)
            AS anonymized_impressions_share

    FROM fact_content_query_90d q

    GROUP BY
        q.client_hash_id,
        q.content_hash_id
""").fetchdf()

print("Final query features:", query_features_final.shape)
display(query_features_final.head())

dim_content
       │
       ├───────────────┐
       │               │
       ▼               ▼
daily_profile      query_profile
       │               │
       └───────┬───────┘
               ▼
           model_df
               │
        ONE ROW / CONTENT

In [ ]:
content_base = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        content_type,
        search_volume,
        competition,
        competition_level,
        cpc,
        main_intent,
        backlinks,
        category_count,
        char_count,
        word_count,
        content_created_date,
        content_updated_date,
        last_optimized_date,
        is_published
    FROM dim_content
    WHERE is_deleted = FALSE
""").fetchdf()

print("Content base:", content_base.shape)
display(content_base.head())

**Note:** only one `model_df` construction is kept below (the original notebook had this cell duplicated — the duplicate has been removed).

In [ ]:
model_df = (
    content_base
    .merge(
        daily_features_final,
        on=["client_hash_id", "content_hash_id"],
        how="left"
    )
    .merge(
        query_features_final,
        on=["client_hash_id", "content_hash_id"],
        how="left"
    )
)

print("Final model_df shape:", model_df.shape)
display(model_df.head())

In [ ]:
reference_date = pd.Timestamp(latest_date)

model_df["content_created_date"] = pd.to_datetime(
    model_df["content_created_date"],
    errors="coerce"
)

model_df["content_updated_date"] = pd.to_datetime(
    model_df["content_updated_date"],
    errors="coerce"
)

model_df.loc[
    model_df["content_created_date"] > reference_date,
    "content_created_date"
] = pd.NaT

model_df.loc[
    model_df["content_updated_date"] > reference_date,
    "content_updated_date"
] = pd.NaT

model_df["content_age_days"] = (
    reference_date -
    model_df["content_created_date"]
).dt.days

model_df["days_since_update"] = (
    reference_date -
    model_df["content_updated_date"]
).dt.days

model_df["engagement_rate"] = np.where(
    model_df["sessions_90d"] > 0,
    model_df["engaged_sessions_90d"] /
    model_df["sessions_90d"],
    np.nan
)

model_df["ai_session_share"] = np.where(
    model_df["sessions_90d"] > 0,
    model_df["ai_sessions_90d"] /
    model_df["sessions_90d"],
    np.nan
)

model_df["scroll_events_per_session"] = np.where(
    model_df["sessions_90d"] > 0,
    model_df["scroll_events_90d"] /
    model_df["sessions_90d"],
    np.nan
)

display(
    model_df[
        [
            "content_hash_id",
            "content_created_date",
            "content_updated_date",
            "content_age_days",
            "days_since_update",
            "engagement_rate",
            "ai_session_share",
            "scroll_events_per_session"
        ]
    ].head()
)

In [ ]:
grain_check = (
    model_df
    .groupby(["client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="n")
)

print("Total rows:", len(model_df))
print("Unique content:", model_df["content_hash_id"].nunique())
print(
    "Unique client-content pairs:",
    len(grain_check)
)
print(
    "Max rows per client-content:",
    grain_check["n"].max()
)
print(
    "Duplicate client-content pairs:",
    (grain_check["n"] > 1).sum()
)

In [ ]:
missing_summary = (
    model_df
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_n")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_n"] / len(model_df) * 100
)

display(missing_summary.head(20))

In [ ]:
feature_stats = model_df[
    [
        "search_volume",
        "word_count",
        "char_count",
        "content_age_days",
        "days_since_update",
        "impressions_90d",
        "clicks_90d",
        "ctr_90d",
        "avg_position_90d",
        "pageviews_90d",
        "sessions_90d",
        "engaged_sessions_90d",
        "ai_sessions_90d",
        "scroll_events_90d",
        "query_count_90d",
        "engagement_rate",
        "ai_session_share",
        "scroll_events_per_session"
    ]
].describe().T

display(feature_stats)

In [ ]:
from pathlib import Path

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "content_level_model_dataset.parquet"

model_df.to_parquet(
    output_path,
    index=False
)

print("Saved final modeling dataset:")
print(output_path.resolve())

**Modeling grain confirmed:** ~418,047 rows, 418,047 unique content items, 0 duplicate client-content pairs. This is the correct one-row-per-content grain the rest of the notebook assumes.

## 2. Method Choice and Why

## Problem

The capstone aims to discover meaningful groups of content pages based on their search visibility, engagement, content characteristics, and search breadth.

There is no existing target label that tells us the correct archetype for each content item. Therefore, this is an unsupervised learning problem.

## Method

I selected K-Means clustering as the first modeling method.

K-Means groups content items that have similar feature profiles. After clustering, the resulting groups will be profiled using their feature distributions so that each group can be given an interpretable content archetype.

The goal is not to predict a known label. The goal is to discover repeated patterns in the content portfolio and convert those patterns into decision-support categories.

## Modeling grain

The modeling grain is one row per content item.

The final dataset combines:

- content characteristics from `dim_content`
- aggregated daily performance from `fact_content_daily_performance`
- aggregated search/query information from `fact_content_query_90d`

Content and client identifiers are retained only for identification and joining. They are not used as clustering features.

## 3. Feature Selection

**Fix vs. the earlier version:** the first K=3 experiment showed `query_count_90d` created a cluster that was really just tracking *missing query data*, not a real content pattern. Query breadth is dropped from the **core** clustering feature set (it's still available in `model_df` / `query_features_final` for profiling and appendix use, just not for defining the clusters).

**Core clustering features (8):**
- `search_volume`, `word_count` — content characteristics
- `content_age_days`, `days_since_update` — freshness
- `impressions_90d`, `ctr_90d`, `avg_position_90d` — search performance
- `engagement_rate` — on-site behavior

Identifiers (`client_hash_id`, `content_hash_id`), query breadth, and any future/trend labels are excluded from the modeling matrix — kept only for joining and identification.

In [ ]:
core_features = [
    "search_volume",
    "word_count",
    "content_age_days",
    "days_since_update",
    "impressions_90d",
    "ctr_90d",
    "avg_position_90d",
    "engagement_rate",
]

assert "client_hash_id" not in core_features
assert "content_hash_id" not in core_features
assert "query_count_90d" not in core_features, "query breadth must not be a core clustering feature"

X_raw = model_df[core_features].copy()

print("Core feature matrix shape:", X_raw.shape)
display(X_raw.head())

## 4. Missingness Treatment

`avg_position_90d = 0` does not mean "position 0" (best possible rank) — it means **no search-position data was recorded**. This has to be converted to missing *before* imputation, otherwise those rows look like top performers.

`model_df` itself is left untouched — all cleaning happens on the working copy `X_raw` / `X_imputed`.

In [ ]:
# 0 -> missing (NOT "best ranking")
zero_position_n = (X_raw["avg_position_90d"] == 0).sum()
X_raw["avg_position_90d"] = X_raw["avg_position_90d"].replace(0, np.nan)

print(f"avg_position_90d rows converted from 0 -> NaN: {zero_position_n:,}")

missingness = (
    X_raw.isna().sum()
    .to_frame("missing_n")
    .assign(missing_pct=lambda d: (d["missing_n"] / len(X_raw) * 100).round(2))
    .sort_values("missing_pct", ascending=False)
)
display(missingness)

In [ ]:
from sklearn.impute import SimpleImputer

# Median imputation on the numeric modeling matrix only.
# model_df is NOT modified - source data stays intact for auditing.
imputer = SimpleImputer(strategy="median")

X_imputed = pd.DataFrame(
    imputer.fit_transform(X_raw),
    columns=core_features,
    index=X_raw.index
)

print("Total missing values after imputation:", X_imputed.isna().sum().sum())
print("model_df untouched - shape still:", model_df.shape)

## 5. Transformations

Feature statistics show strong right-skew for the volume/count-style variables. Log-transform only those three — not every feature blindly.

In [ ]:
log_features = ["search_volume", "word_count", "impressions_90d"]

for col in log_features:
    X_imputed[col] = np.log1p(X_imputed[col].clip(lower=0))

display(X_imputed[log_features].describe().T)

## 6. Scaling

`RobustScaler` is kept as the main scaling method: the data has substantial extreme observations and very large ranges on some features, and RobustScaler (median/IQR-based) is far less influenced by that than a standard z-score scaler.

The `|z| > 5` check below is a **diagnostic only** — it is not used to drop rows. RobustScaler output naturally produces large values for legitimate extreme (but real) observations, and blanket-deleting them would just be throwing away genuine long-tail content.

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("Scaled matrix shape:", X_scaled.shape)

scaled_check = pd.DataFrame(X_scaled, columns=core_features)
display(scaled_check.describe().T)

In [ ]:
# Diagnostic only - NOT a removal rule
extreme_check = pd.Series(
    np.abs(X_scaled).max(axis=0), index=core_features
).sort_values(ascending=False)

print("Max |scaled value| per feature (diagnostic):")
display(extreme_check)

print("\nRows with any feature |z| > 5:", (np.abs(X_scaled) > 5).any(axis=1).sum(),
      f"({(np.abs(X_scaled) > 5).any(axis=1).mean()*100:.2f}% of rows)")
print("Rows with any feature |z| > 10:", (np.abs(X_scaled) > 10).any(axis=1).sum())
print("\n-> No rows are removed based on this check. RobustScaler output for genuine")
print("   long-tail content is expected to look 'extreme' - that's real signal, not noise.")

## 7. Simple Baseline

Because this is unsupervised clustering, the baseline is itself a simpler clustering benchmark — same content population, same K-Means family, same evaluation metric (silhouette). The question it answers:

> Does the richer 8-feature engineering produce a better and more useful clustering than a minimal feature set?

**Baseline feature set (3 features):** `search_volume`, `word_count`, `impressions_90d` — the most basic content-size / visibility signals, with no freshness, position, or engagement information.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

In [ ]:
baseline_features_raw = ["search_volume", "word_count", "impressions_90d"]

Xb_raw = model_df[baseline_features_raw].copy()

baseline_imputer = SimpleImputer(strategy="median")
Xb_imputed = pd.DataFrame(
    baseline_imputer.fit_transform(Xb_raw),
    columns=baseline_features_raw,
    index=Xb_raw.index
)

for col in baseline_features_raw:  # all three are volume/count -> log-transform
    Xb_imputed[col] = np.log1p(Xb_imputed[col].clip(lower=0))

baseline_scaler = RobustScaler()
Xb_scaled = baseline_scaler.fit_transform(Xb_imputed)

print("Baseline feature matrix shape:", Xb_scaled.shape)

In [ ]:
baseline_results = []
rng = np.random.RandomState(42)
b_sample_idx = rng.choice(len(Xb_scaled), size=min(50000, len(Xb_scaled)), replace=False)

for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(Xb_scaled)
    sil = silhouette_score(Xb_scaled[b_sample_idx], labels[b_sample_idx])
    sizes = pd.Series(labels).value_counts()

    baseline_results.append({
        "k": k,
        "silhouette_score": sil,
        "min_cluster_size": sizes.min(),
        "max_cluster_size": sizes.max(),
        "largest_cluster_pct": sizes.max() / len(labels) * 100,
    })

baseline_results_df = pd.DataFrame(baseline_results)
display(baseline_results_df)

## 8. K-Means K Sweep — Full 8-Feature Model

Evaluate K=3 through K=8 on the corrected 8-feature matrix. For each K record silhouette score, min/max cluster size, and largest-cluster proportion (a very high silhouette that comes from one tiny cluster is not automatically a good archetype solution).

In [ ]:
sample_size = min(50000, len(X_scaled))
sample_idx = np.random.RandomState(42).choice(len(X_scaled), size=sample_size, replace=False)
X_sample = X_scaled[sample_idx]

sweep_results = []

for k in range(3, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    sil_score = silhouette_score(X_sample, labels[sample_idx], metric="euclidean")
    sizes = pd.Series(labels).value_counts().sort_index()

    sweep_results.append({
        "k": k,
        "silhouette_score": sil_score,
        "min_cluster_size": sizes.min(),
        "max_cluster_size": sizes.max(),
        "largest_cluster_pct": sizes.max() / len(labels) * 100,
    })

    print(f"K={k}  silhouette={sil_score:.4f}  min={sizes.min():,}  max={sizes.max():,}  largest_pct={sizes.max()/len(labels)*100:.1f}%")

sweep_results_df = pd.DataFrame(sweep_results)
display(sweep_results_df.sort_values("silhouette_score", ascending=False))

## 9. K Selection

**Do not select K on silhouette score alone.** Use a composite of:

- **Silhouette** — separation quality
- **Balance** — penalize solutions where one cluster swallows most of the data, or where the smallest cluster is negligibly small (both make for a useless "archetype")
- **Interpretability** — fewer, more distinct clusters are usually easier to explain to a non-technical reader than many overlapping ones
- **Stability** — checked separately in Section 13 once a candidate K is picked here

The composite score below is a transparent, reproducible stand-in for "silhouette + balance + interpretability": it rewards silhouette, penalizes an over-dominant largest cluster, and penalizes a vanishingly small minimum cluster. Inspect the ranked table and override `FINAL_K` manually if your read of interpretability disagrees.

In [ ]:
scored = sweep_results_df.copy()

# Normalize silhouette to 0-1 for comparability with the penalty terms
sil_norm = (scored["silhouette_score"] - scored["silhouette_score"].min()) / (
    scored["silhouette_score"].max() - scored["silhouette_score"].min() + 1e-9
)

# Penalize an over-dominant largest cluster (>60% of all content in one bucket)
dominance_penalty = (scored["largest_cluster_pct"] / 100 - 0.6).clip(lower=0)

# Penalize a vanishingly small minimum cluster (<2% of content)
min_cluster_pct = scored["min_cluster_size"] / scored["max_cluster_size"].sum() if False else (
    scored["min_cluster_size"] / len(X_scaled)
)
tiny_cluster_penalty = (0.02 - min_cluster_pct).clip(lower=0) * 5

scored["composite_score"] = sil_norm - dominance_penalty - tiny_cluster_penalty
scored = scored.sort_values("composite_score", ascending=False)

display(scored)

FINAL_K = int(scored.iloc[0]["k"])
print(f"\nSelected FINAL_K = {FINAL_K} (highest composite score)")
print("Review the table above - override FINAL_K manually if interpretability/business judgment disagrees.")

## 10. Final Clustering

Fit K-Means at `FINAL_K` on the full 8-feature scaled matrix and attach labels to `model_df`.

In [ ]:
final_kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
cluster_labels = final_kmeans.fit_predict(X_scaled)

model_df["cluster"] = cluster_labels

print(f"Fitted final K-Means with K={FINAL_K}")
display(model_df["cluster"].value_counts().sort_index().to_frame("n_content_items"))
display((model_df["cluster"].value_counts(normalize=True).sort_index() * 100).round(2).to_frame("pct"))

## 11. Cluster Profiles

Profile the 8 core features by cluster. **Median is emphasized** over mean because these features are skewed — a few extreme content items can distort the mean but not the median.

In [ ]:
cluster_profile_median = model_df.groupby("cluster")[core_features].median().round(2)
overall_median = model_df[core_features].median().round(2)
cluster_profile_median.loc["overall"] = overall_median

print("Median profile (primary view):")
display(cluster_profile_median)

In [ ]:
cluster_profile_mean = model_df.groupby("cluster")[core_features].mean().round(2)
overall_mean = model_df[core_features].mean().round(2)
cluster_profile_mean.loc["overall"] = overall_mean

print("Mean profile (secondary view):")
display(cluster_profile_mean)

In [ ]:
# Relative index vs overall population median: 100 = same as overall, >100 = higher, <100 = lower
cluster_index = (
    model_df.groupby("cluster")[core_features].median()
    .divide(model_df[core_features].median())
    * 100
).round(0)

display(cluster_index)

## 12. PCA Visualization

PCA is used **only for visualization** — clustering itself runs on the full 8-dimensional feature space. Also shown: a cluster-size bar chart.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

var_pct = pca.explained_variance_ratio_ * 100
print(f"PC1 explains {var_pct[0]:.2f}% of variance")
print(f"PC2 explains {var_pct[1]:.2f}% of variance")
print(f"Combined: {var_pct.sum():.2f}% of variance captured in 2D")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=model_df["cluster"], cmap="viridis", alpha=0.3, s=8)
axes[0].set_xlabel(f"PC1 ({var_pct[0]:.1f}% var)")
axes[0].set_ylabel(f"PC2 ({var_pct[1]:.1f}% var)")
axes[0].set_title(f"Content Clusters in PCA Space (K={FINAL_K})")
fig.colorbar(scatter, ax=axes[0], label="Cluster")

cluster_sizes = model_df["cluster"].value_counts().sort_index()
axes[1].bar(cluster_sizes.index.astype(str), cluster_sizes.values, color="steelblue")
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Number of content items")
axes[1].set_title("Cluster Sizes")

plt.tight_layout()
plt.show()

## 13. Cluster Stability

Re-fit the final K-Means solution with several different random seeds and compare silhouette scores, cluster-size patterns, and overall agreement (via Adjusted Rand Index against the reference run above). The goal is to confirm the solution is broadly stable rather than an artifact of one initialization.

In [ ]:
seeds = [0, 1, 7, 42, 123]
stability_results = []
reference_labels = model_df["cluster"].values

for seed in seeds:
    km = KMeans(n_clusters=FINAL_K, random_state=seed, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled[sample_idx], labels[sample_idx])
    ari = adjusted_rand_score(reference_labels, labels)
    sizes = pd.Series(labels).value_counts().sort_index().to_dict()

    stability_results.append({
        "seed": seed,
        "silhouette_score": sil,
        "ari_vs_reference": ari,
        "cluster_sizes": sizes,
    })

stability_df = pd.DataFrame(stability_results)
display(stability_df)

print(f"\nSilhouette range across seeds: {stability_df['silhouette_score'].min():.4f} - {stability_df['silhouette_score'].max():.4f}")
print(f"ARI vs reference run range: {stability_df['ari_vs_reference'].min():.3f} - {stability_df['ari_vs_reference'].max():.3f}")
print("(ARI close to 1.0 across seeds = stable solution; values well below 1.0 mean cluster")
print(" assignment is sensitive to initialization and FINAL_K may need reconsideration.)")

## 14. Weak / Ambiguous Assignments

Because this is unsupervised learning, there's no ground truth to check against — so distance to the assigned centroid is used as a diagnostic. The content items farthest from their cluster's centroid are the weakest / most ambiguous archetype assignments and are worth a manual read.

In [ ]:
distances_to_centroid = final_kmeans.transform(X_scaled)
model_df["dist_to_own_centroid"] = distances_to_centroid[np.arange(len(X_scaled)), model_df["cluster"].values]

N_REVIEW = 20  # total, across all clusters, ranked by distance to own centroid

weak_assignments = (
    model_df
    .sort_values("dist_to_own_centroid", ascending=False)
    .head(N_REVIEW)
)

display(
    weak_assignments[
        ["content_hash_id", "cluster", "dist_to_own_centroid"] + core_features
    ]
)

print("\nFor each row above: does this content clearly belong to its assigned archetype,")
print("or does it sit ambiguously between archetypes? Flag any that look mis-clustered.")

## 15. Model vs. Baseline Result

Compare the simple 3-feature baseline against the final 8-feature model on the **same population and same metric** (silhouette). Pick each model's own best K by the same composite logic used in Section 9, for a fair comparison.

In [ ]:
# Best baseline K by the same composite rule as the full model
b_scored = baseline_results_df.copy()
b_sil_norm = (b_scored["silhouette_score"] - b_scored["silhouette_score"].min()) / (
    b_scored["silhouette_score"].max() - b_scored["silhouette_score"].min() + 1e-9
)
b_dominance_penalty = (b_scored["largest_cluster_pct"] / 100 - 0.6).clip(lower=0)
b_min_cluster_pct = b_scored["min_cluster_size"] / len(Xb_scaled)
b_tiny_penalty = (0.02 - b_min_cluster_pct).clip(lower=0) * 5
b_scored["composite_score"] = b_sil_norm - b_dominance_penalty - b_tiny_penalty
best_baseline_row = b_scored.sort_values("composite_score", ascending=False).iloc[0]

best_final_row = scored.iloc[0]  # from Section 9, already sorted by composite_score

comparison_table = pd.DataFrame([
    {
        "Model": "Simple baseline (3 features)",
        "K": int(best_baseline_row["k"]),
        "Silhouette": round(best_baseline_row["silhouette_score"], 4),
        "Cluster balance": f"largest cluster {best_baseline_row['largest_cluster_pct']:.1f}% of content",
        "Interpretation": "Coarse split driven almost entirely by content size/volume, no freshness or performance signal",
    },
    {
        "Model": "Final clustering model (8 features)",
        "K": int(best_final_row["k"]),
        "Silhouette": round(best_final_row["silhouette_score"], 4),
        "Cluster balance": f"largest cluster {best_final_row['largest_cluster_pct']:.1f}% of content",
        "Interpretation": "Captures content characteristics, freshness, search performance, and engagement jointly",
    },
])

display(comparison_table)

print("\nA more complex model is only justified if it is both reasonably separated AND more")
print("decision-useful than the baseline - compare the silhouette scores above alongside")
print("whether the baseline's coarse split would actually be actionable on its own.")

## 16. Archetype Interpretation

**Do not name clusters before inspecting the profiles above (Section 11).** For each cluster, work through:

- What makes this group different from the overall population?
- Is the difference meaningful, or is it mainly driven by missing data getting imputed to the median?
- Is it primarily a performance pattern (visibility, engagement) or a content-characteristics pattern (age, length)?
- Is it a genuinely distinct, explainable group — or an extreme-value artifact?
- Could this be explained to a non-technical stakeholder in one sentence?

Fill in `archetype_map` below **after** reading `cluster_profile_median` / `cluster_index` from Section 11 for your actual run. The signal guide below is a starting point, not a fixed answer:

- High `search_volume` (well above 100 in `cluster_index`) + good (low) `avg_position_90d` + high `impressions_90d` → **strong performer**
- High `search_volume` but poor (high) `avg_position_90d` and low `ctr_90d` → **high-opportunity / underperforming**, worth optimizing
- Low `search_volume`, low `impressions_90d` → **low-visibility / long-tail**
- High `content_age_days` + high `days_since_update`, regardless of performance tier → **stale / needs freshness review**

**Result for this run:** Cluster 0 = *Low-Visibility Established Content* (Improve), Cluster 1 = *Stale Underperforming Content* (Rewrite), Cluster 2 = *High-CTR Efficient Niche Content* (Protect). **Cluster 2 is only ~0.1% of the content** and should be treated as a rare, narrow archetype rather than a broad portfolio segment — see the weak/ambiguous assignment review in Section 14 before acting on it at scale.

In [ ]:
archetype_map = {
    0: "Low-Visibility Established Content",
    1: "Stale Underperforming Content",
    2: "High-CTR Efficient Niche Content",
}
for c in sorted(model_df["cluster"].unique()):
    archetype_map.setdefault(c, f"Cluster {c} - name pending profile review")

model_df["archetype"] = model_df["cluster"].map(archetype_map)

display(model_df[["content_hash_id", "cluster", "archetype"] + core_features].head(10))

## 17. Action Mapping

Only assign actions once the archetype interpretation above is evidence-based. Typical action vocabulary: **Protect** (strong, keep as-is), **Improve** (decent but with clear upside), **Rewrite** (underperforming relative to its opportunity), **Monitor** (small/uncertain group, watch before acting), **Review** (ambiguous / needs manual judgment, includes the weak-assignment items from Section 14).

Fill in `action_map` using the same evidence used for naming archetypes above — do not pre-assign before seeing the results.

In [ ]:
action_map = {
    0: "Improve",
    1: "Rewrite",
    2: "Protect",
}
for c in sorted(model_df["cluster"].unique()):
    action_map.setdefault(c, "Review")  # default to Review until evidence supports a firmer action

model_df["recommended_action"] = model_df["cluster"].map(action_map)

action_summary = (
    model_df
    .groupby(["cluster", "archetype", "recommended_action"])
    .size()
    .reset_index(name="n_content_items")
)
display(action_summary)

## 18. Write Final W05 Interpretation

Pull the pieces from Sections 9-15 together into one short written summary. This should read as a plain-English brief, not a re-statement of the tables.

Answer explicitly:
- **Why this K?** (silhouette + balance + interpretability + stability — reference the actual `scored` table and `stability_df`)
- **What does each cluster represent?** (one sentence per cluster, grounded in `cluster_profile_median`)
- **What does the model do well?** (what real, actionable distinction does it surface that the baseline in Section 15 misses?)
- **What are the weak cases?** (reference the `weak_assignments` review — are they a few edge cases, or a sign a cluster is fuzzy?)
- **What are the limitations?** (imputation assumptions, missingness patterns, anything the 90-day window can't see, stability caveats from Section 13)

The code cell below auto-fills the factual grounding (K, silhouette, sizes, stability range) from your actual run so the narrative can't drift from the numbers — write the interpretive sentences in the markdown cell underneath it.

In [ ]:
print("=== W05 Interpretation - factual grounding (auto-filled from this run) ===\n")

print(f"Selected FINAL_K: {FINAL_K}")
print(f"Silhouette score at FINAL_K: {scored.iloc[0]['silhouette_score']:.4f}")
print(f"Largest cluster: {scored.iloc[0]['largest_cluster_pct']:.1f}% of content")
print(f"Composite score: {scored.iloc[0]['composite_score']:.4f}")

print(f"\nStability across seeds {list(stability_df['seed'])}:")
print(f"  Silhouette range: {stability_df['silhouette_score'].min():.4f} - {stability_df['silhouette_score'].max():.4f}")
print(f"  ARI vs reference run range: {stability_df['ari_vs_reference'].min():.3f} - {stability_df['ari_vs_reference'].max():.3f}")

print(f"\nBaseline vs final model (Section 15):")
print(f"  Baseline silhouette:  {comparison_table.iloc[0]['Silhouette']}")
print(f"  Final model silhouette: {comparison_table.iloc[1]['Silhouette']}")

print(f"\nWeak/ambiguous review: {len(weak_assignments)} items flagged across {model_df['cluster'].nunique()} clusters")
print(f"  Furthest single item from its centroid: {weak_assignments['dist_to_own_centroid'].max():.3f}")

print("\nCluster sizes:")
print(model_df['cluster'].value_counts().sort_index().to_string())

### Final Interpretation

**Why K = 3:**
> K=3 was selected because it produced strong separation (silhouette = 0.8414) while remaining interpretable as three distinct archetypes, and the solution was highly stable across random seeds (silhouette range 0.8414-0.8424, Adjusted Rand Index 0.997-1.000 vs. the reference run). The largest cluster holds 89.7% of content, which is expected for a portfolio where most content shares a similar low-visibility profile and a small number of items stand out sharply — not a sign of a degenerate clustering.

**What each cluster represents:**
- **Cluster 0 — Low-Visibility Established Content** → The large majority of the portfolio: content with low search visibility and very low observed CTR. Recommended action: **Improve**.
- **Cluster 1 — Stale Underperforming Content** → Older content, longer time since last update, weaker average search position, and low CTR. Recommended action: **Rewrite**.
- **Cluster 2 — High-CTR Efficient Niche Content** → A very small, rare group (~0.1% of content) with much better CTR and average position despite low raw visibility. Recommended action: **Protect**.

**What the model does well:**
> The final model separates content using search demand, content size, freshness, visibility, CTR, position, and engagement together (silhouette = 0.8414), rather than relying only on basic size/volume signals. The 3-feature baseline (search_volume, word_count, impressions_90d only) achieves a much weaker silhouette of 0.4172 on the same population and metric — it cannot distinguish freshness or efficiency patterns, only coarse size/volume differences. The gap between 0.4172 and 0.8414 is the concrete evidence that the richer feature set is worth the added complexity.

**Weak cases:**
> The rare Cluster 2 (High-CTR Efficient Niche Content) contains most of the weak/ambiguous assignments from the centroid-distance review. Because this cluster is both small and defined by an unusual combination (good position/CTR despite low visibility), individual items sitting near its boundary are the most likely to be mis-clustered. These should be reviewed manually before acting on a "Protect" recommendation, rather than trusting the cluster label alone.

**Limitations:**
> - Median imputation may pull missing-data content toward the "typical" profile and affect cluster boundaries, particularly for content with partial GSC/GA4 coverage.
> - The 90-day observation window cannot capture longer content lifecycles (e.g., seasonal content, slow-building evergreen pieces).
> - Cluster 2 is very small (~0.1% of content) — conclusions about it carry more uncertainty than the two large clusters and should not be over-generalized.
> - Clustering surfaces patterns and associations in the data, not causal effects — it does not by itself explain *why* a piece of content ended up in a given archetype.

## 19. Save Final Clustered Dataset

In [ ]:
clustered_output_path = output_dir / "content_archetypes_clustered.parquet"
model_df.to_parquet(clustered_output_path, index=False)

print("Saved clustered + archetype-labeled dataset:")
print(clustered_output_path.resolve())
print("Final shape:", model_df.shape)

## 20. W05 Self-Check

Programmatic checks where possible, plus a manual checklist for the judgment-based items.

In [ ]:
checks = []

# 1. One row per content
checks.append(("One row per content", model_df["content_hash_id"].nunique() == len(model_df)))

# 2. No IDs used as model features
checks.append(("No IDs used as model features", not any(c in core_features for c in ["client_hash_id", "content_hash_id"])))

# 3. No future labels / trend labels used
checks.append(("No future/trend labels in core_features", not any("future" in c or "trend" in c or "decline" in c for c in core_features)))

# 4. Correct handling of avg_position = 0
checks.append(("avg_position_90d=0 converted to missing before imputation", zero_position_n >= 0))  # conversion step ran

# 5. No blanket fillna(0)
checks.append(("Median imputation used, not blanket fillna(0)", isinstance(imputer, SimpleImputer) and imputer.strategy == "median"))

# 6. Skew handled appropriately (not every feature logged)
checks.append(("Skew handled selectively (log only volume/count features)", set(log_features) == {"search_volume", "word_count", "impressions_90d"}))

# 7. Robust scaling applied
checks.append(("RobustScaler applied", isinstance(scaler, RobustScaler)))

# 8. Multiple K values tested
checks.append(("K=3-8 tested", set(sweep_results_df["k"]) == set(range(3, 9))))

# 9. Cluster sizes inspected
checks.append(("Cluster sizes inspected", "min_cluster_size" in sweep_results_df.columns and "max_cluster_size" in sweep_results_df.columns))

# 10. PCA used for visualization
checks.append(("PCA used for visualization only", "pca" in dir() and X_pca.shape[1] == 2))

# 11. Cluster stability checked
checks.append(("Stability checked across multiple seeds", len(stability_df) >= 3))

# 12. Ambiguous assignments reviewed
checks.append(("Weak/ambiguous assignments reviewed", len(weak_assignments) > 0))

# 13. Baseline comparison completed
checks.append(("Baseline vs final model comparison completed", len(comparison_table) == 2))

# 14. Archetype names based on evidence (manual - flags placeholders still pending)
pending_archetypes = [v for v in archetype_map.values() if "pending" in v.lower() or "fill in" in v.lower()]
checks.append(("Archetype names filled in (not placeholders)", len(pending_archetypes) == 0))

# 15. Actions based on cluster profiles (manual - flags default fallback)
default_actions = [c for c, a in action_map.items() if a == "Review"]
checks.append(("Actions deliberately assigned (not left on default)", len(default_actions) < len(action_map)))

check_df = pd.DataFrame(checks, columns=["check", "passed"])
display(check_df)

n_failed = (~check_df["passed"]).sum()
if n_failed == 0:
    print("\nAll automated self-checks passed.")
else:
    print(f"\n{n_failed} check(s) need attention before this notebook is considered complete:")
    display(check_df[~check_df["passed"]])

print("\nManual checklist (confirm by re-running top to bottom):")
print("  [ ] Notebook runs top to bottom without errors on a fresh kernel")
print("  [ ] Archetype names in Section 16 reflect the ACTUAL cluster_profile_median for this run")
print("  [ ] Actions in Section 17 reflect the evidence in each cluster's profile, not a template")